# Object Numeric-String Contract v0.2.2

**This is the final bounded contract patch before Stage 0 Data Audit.**

Bootstrap có thể clone/fetch repository. Bản thân survey không scan thêm dữ liệu, không decode media, không chạy model/GPU và không sửa source JSON.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
DATASET_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
OUTPUT_ROOT = Path('/kaggle/working/object_numeric_contract_v022')
V021_SUMMARY = Path('/kaggle/working/cross_asset_survey_v021/patch_summary_v021.json')
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
print('This is the final bounded contract patch before Stage 0 Data Audit.')
print('repo:', REPO_URL, 'ref:', REPO_REF)
print('dataset:', DATASET_ROOT)
print('output:', OUTPUT_ROOT)

In [ ]:
def git(*args, cwd=None):
    completed = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return completed.stdout.strip()

module_file = REPO_DIR / 'src/triage_eg/data/object_numeric_contract.py'
if (REPO_DIR / '.git').is_dir():
    if REFRESH_REPO or not module_file.is_file():
        git('fetch', '--depth', '1', 'origin', REPO_REF, cwd=REPO_DIR)
        git('checkout', '--detach', 'FETCH_HEAD', cwd=REPO_DIR)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout. Set AIC_REPO_DIR to an empty path and rerun.')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git('clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR))
    git('fetch', '--depth', '1', 'origin', REPO_REF, cwd=REPO_DIR)
    git('checkout', '--detach', 'FETCH_HEAD', cwd=REPO_DIR)
if not module_file.is_file():
    raise RuntimeError(f'{module_file} is absent from remote ref {REPO_REF!r}. Commit and push v0.2.2 to that branch, then set AIC_REFRESH_REPO=1 and rerun.')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('git commit:', git('rev-parse', 'HEAD', cwd=REPO_DIR))
print('python:', sys.version)

In [ ]:
from triage_eg.data.object_numeric_contract import NumericLimits, run_survey, write_outputs
limits = NumericLimits(max_object_json_total=15, max_object_json_bytes=1048576)
result = run_survey(DATASET_ROOT, limits=limits, strict_root=True, v021_summary=V021_SUMMARY if V021_SUMMARY.is_file() else None)
artifact_paths = write_outputs(result, OUTPUT_ROOT)
print(result.summary['disclaimer'])

In [ ]:
coordinate_summary = result.summary['coordinate_summary']
print({k:v for k,v in coordinate_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
score_summary = result.summary['score_summary']
print({k:v for k,v in score_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
label_summary = result.summary['label_summary']
print({k:v for k,v in label_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
print('invalid sample count:', len(result.invalid_samples))
for sample in result.invalid_samples[:20]: print(sample)

In [ ]:
print(result.summary['normalization_contract'])

In [ ]:
print(result.summary['readiness'])
print(result.summary['issues_summary'])

In [ ]:
from zipfile import ZipFile
zip_path = artifact_paths['zip']
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert len(members) == 5
assert 'object_numeric_contract_v022.zip' not in members
print('ZIP verified:', members)

In [ ]:
print('DOWNLOAD ZIP:', zip_path)
print('size_bytes:', zip_path.stat().st_size)